# 1972 Prolog

Alain Colmerauer, Philippe Roussel, and collaborators created Prolog in 1972 as a practical logic programming language. It became important to AI because programs could be written as facts and rules, then queried by asking the system what follows from those statements.

## High-Level Ideas

Prolog programs are built from a small set of logic programming pieces:

- **Facts** state things that are known to be true, such as `father_child(tom, sally)`.
- **Rules** define relationships that can be inferred from other facts or rules.
- **Queries** ask Prolog to search for values that make a statement true.
- **Unification** matches a query against facts and rules, binding variables along the way.

A rule has the form:

```prolog
conclusion :- condition_1, condition_2.
```

Read `:-` as "if". The conclusion is true if every condition to the right of `:-` can be proven true.

In [1]:
prolog_source = r"""
mother_child(trude, sally).

father_child(tom, sally).
father_child(tom, erica).
father_child(mike, tom).

parent_child(Parent, Child) :- mother_child(Parent, Child).
parent_child(Parent, Child) :- father_child(Parent, Child).

sibling(Person, Sibling) :- parent_child(Parent, Person), parent_child(Parent, Sibling), Person \= Sibling.

ancestor(Ancestor, Descendant) :- parent_child(Ancestor, Descendant).
ancestor(Ancestor, Descendant) :- parent_child(Ancestor, Person), ancestor(Person, Descendant).
"""

print(prolog_source)


mother_child(trude, sally).

father_child(tom, sally).
father_child(tom, erica).
father_child(mike, tom).

parent_child(Parent, Child) :- mother_child(Parent, Child).
parent_child(Parent, Child) :- father_child(Parent, Child).

sibling(Person, Sibling) :- parent_child(Parent, Person), parent_child(Parent, Sibling), Person \= Sibling.

ancestor(Ancestor, Descendant) :- parent_child(Ancestor, Descendant).
ancestor(Ancestor, Descendant) :- parent_child(Ancestor, Person), ancestor(Person, Descendant).



## Querying Prolog from Python

PySwip connects Python to SWI-Prolog. PySwip can be installed as a Python package, but it still needs SWI-Prolog available on the machine as the underlying Prolog runtime.

To run the PySwip cells below, install SWI-Prolog first. See the PySwip setup guide: https://pyswip.readthedocs.io/en/stable/get_started.html#installing-swi-prolog

In [2]:
swi_prolog_install_url = "https://pyswip.readthedocs.io/en/stable/get_started.html#installing-swi-prolog"

try:
    from pyswip import Prolog
except Exception:
    Prolog = None


def load_program(source):
    if Prolog is None:
        return None

    prolog = Prolog()
    for statement in source.splitlines():
        statement = statement.strip()
        if not statement:
            continue
        prolog.assertz(statement.removesuffix("."))
    return prolog


prolog = load_program(prolog_source)

In [3]:
queries = {
    "children of tom": "father_child(tom, Child)",
    "siblings of sally": "sibling(sally, Sibling)",
    "ancestors of sally": "ancestor(Ancestor, sally)",
    "descendants of mike": "ancestor(mike, Descendant)",
}

if prolog is None:
    print("SWI-Prolog is not installed. Install it before running these PySwip queries:")
    print(swi_prolog_install_url)
else:
    for label, query in queries.items():
        answers = list(prolog.query(query))

        print(f"{label}:")
        for answer in answers:
            print(f"  {answer}")

children of tom:
  {'Child': 'sally'}
  {'Child': 'erica'}
siblings of sally:
  {'Sibling': 'erica'}
ancestors of sally:
  {'Ancestor': 'trude'}
  {'Ancestor': 'tom'}
  {'Ancestor': 'mike'}
descendants of mike:
  {'Descendant': 'tom'}
  {'Descendant': 'sally'}
  {'Descendant': 'erica'}


## Why This Mattered

Prolog made symbolic AI feel executable. Instead of spelling out every search step, a programmer could declare relationships and let the language search for proofs. This style fit expert systems, theorem proving, natural language processing experiments, and other AI work where explicit rules were easier to inspect than numeric weights.